# GTEx colocalization with 1000 Genomes LD

In [8]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/code/Gtex_fine.R
# ============================================================
# GTEx eQTL fine-mapping with LD sketch reference
#   - SuSiE (individual)
#   - SuSiE-RSS with sketch (no EB)
#   - SuSiE-RSS with sketch + R_finite EB correction
#
# Usage: Rscript KG_fine.R <IDX>
#   IDX is the SLURM array task id; each task processes 50 genes
#   from /home/sunh4/rss_qc/gtex/GTEx_genes_5000.txt
# ============================================================

# ---- batch index from command line ----
args <- commandArgs(trailingOnly = TRUE)
IDX  <- as.integer(args[1])
stopifnot(!is.na(IDX), IDX >= 1)

# ---- libraries ----
library(susieR)
library(data.table)
library(tidyverse)
library(BEDMatrix)
library(vroom)
library(mixsqp)
library(pecotmr)
library(pgenlibr)
library(matrixStats)
library(crayon)
library(Rfast)

# ---- local source overrides ----
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# ============================================================
# Two susieR builds, each ISOLATED in its own environment so their
# identically-named functions never clobber.
#   susie_hc / susie_indiv : haochen build (susie_rss + individual-data susie)
#   susie_xw               : xuewei build  (R_mismatch = "eb_mix")
# ============================================================
source_build <- function(dir) {
  rdir <- if (dir.exists(file.path(dir, "R"))) file.path(dir, "R") else dir
  env  <- new.env(parent = globalenv())
  for (f in list.files(rdir, full.names = TRUE, pattern = "\\.R$"))
    sys.source(f, envir = env)
  env
}

susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss
susie_xw <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/susieR_July17_2026_singleL/")$susie_rss
susie_indiv  <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie
# ---- helpers ----
safe_susie <- function(.fn, .label, ...) {
  tryCatch(
    withCallingHandlers(
      .fn(...),
      warning = function(w) invokeRestart("muffleWarning")
    ),
    error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
      NULL
    }
  )
}

# max inter-CS correlation (NA when < 2 CS), against the LD the fit used
cs_maxcor <- function(model, Xcorr)
  tryCatch(as.numeric(get_cs_correlation(model, Xcorr = Xcorr, max = TRUE)),
           error = function(e) NA_real_)

sketch_harmonize <- function(target_data, ref_variants, geno_1kg) {
  qc_data <- allele_qc(match_min_prop = 0,
                       target_data    = target_data,
                       ref_variants   = ref_variants)
  qced <- qc_data$target_data_qced
  idx  <- match(qced$variant_id, colnames(geno_1kg))
  if (anyNA(idx)) stop("Some qced variant_id values not found in 1000G columns.")
  geno2 <- geno_1kg[, idx, drop = FALSE]
  flip_index <- which(qced$variants_id_original != qced$variant_id)
  if (length(flip_index) > 0)
    geno2[, flip_index] <- -geno2[, flip_index, drop = FALSE]
  colnames(geno2) <- qced$variants_id_original
  list(sumstat = qced, geno = geno2,
       flipped_prop = length(flip_index) / ncol(geno2))
}

kriging_fix <- function(z, LD, n, z_thres = 2, logLR_thres = 2) {
  krig <- kriging_rss(z = z, R = LD, n = n)
  krig_outlier <- krig$conditional_dist %>%
    mutate(index = row_number()) %>%
    filter(abs(z) > z_thres & logLR > logLR_thres)
  z_fix <- z
  z_fix[krig_outlier$index] <- -z_fix[krig_outlier$index]
  list(z = z_fix,
       outlier_index  = krig_outlier$index,
       outlier_number = length(krig_outlier$index),
       output         = krig$conditional_dist)
}

# ---- config ----
geneTSS_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list"
genotype_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/genotype"
eqtl_dir     <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/normalized_expression"
cov_dir      <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/GTEx/GTEx_Analysis_v8_eQTL_covariates"
ld_1kg_dir   <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_EUR_hg38"   # BED/BIM/FAM per chr
gene_list    <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/gtex/GTEx_genes_5000.txt"
out_dir      <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/res"


maf.cut            <- 0.05
missing.cut        <- 0.1
tissues_considered <- NULL
data.table         <- FALSE
B_insample  <- 1e6      # in-sample LD is exact -> huge R_finite (EB captures only residual)
n_kg         <- 489     # 1000G EUR sample size for R_finite on KG LD

dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

GTEx_genes <- read.table(gene_list, header = TRUE)
idx_start  <- 50 * (IDX - 1) + 1
idx_end    <- min(50 * IDX, nrow(GTEx_genes))

message(sprintf("[IDX=%d] processing genes %d-%d of %d",
                IDX, idx_start, idx_end, nrow(GTEx_genes)))

# ============================================================
# Outer loop: genes
# ============================================================

geneTSS <- read_delim(geneTSS_dir, show_col_types = FALSE)

for (i in idx_start:idx_end) {

  gene.name <- GTEx_genes$x[i]

  # skip-existing
  out_path <- file.path(out_dir, paste0("res_", gene.name, ".rds"))
  if (file.exists(out_path)) next
  print(i)

  message(sprintf("[%s] === gene %d/%d: %s ===",
                  format(Sys.time(), "%Y-%m-%d %H:%M:%S"),
                  i, idx_end, gene.name))

  # ---- 1. gene TSS / region ----
  gene.TSS <- geneTSS %>%
    filter(gene_id == gene.name) %>%
    mutate(CHR        = str_extract(`#chr`, "(?<=chr)\\d+"),
           gene_start = pmax(0, start - 1e6),
           gene_end   = start + 1e6) %>%
    slice(1)

  if (nrow(gene.TSS) == 0) {
    saveRDS("gene not found", file.path(out_dir, paste0("nores_", gene.name, ".rds")))
    next
  }

  chr        <- gene.TSS$CHR
  gene.start <- gene.TSS$gene_start
  gene.end   <- gene.TSS$gene_end

      # skip chrX (no 1000G LD reference available)
    if (is.na(chr)) {
      message(sprintf("  [%s] Skipping chrX", gene.name))
      next
    }

  # ---- 2. GTEx individual genotype ----
  message(sprintf("  [%s] Loading GTEx genotype (chr %s)", gene.name, chr))
  genotype <- BEDMatrix(file.path(genotype_dir, paste0("GTEx_v8_genotype.", chr)))

  bim_data <- {
    path <- file.path(genotype_dir, paste0("GTEx_v8_genotype.", chr, ".bim"))
    if (data.table) read_delim(path, header = FALSE)
    else            read.delim(path, header = FALSE, stringsAsFactors = FALSE)
  }

  gene.snp <- bim_data %>%
    mutate(idx = row_number()) %>%
    filter(V4 >= gene.start & V4 <= gene.end)

  if (nrow(gene.snp) == 0) {
    saveRDS("no SNPs in region", file.path(out_dir, paste0("nores_", gene.name, ".rds")))
    next
  }

  geno_mat <- data.matrix(genotype[, gene.snp$idx])
  rownames(geno_mat) <- str_replace(rownames(geno_mat), "0_", "")
  colnames(geno_mat) <- paste0("chr", gene.snp$V1, ":", gene.snp$V4, ":",
                               gene.snp$V6, ":", gene.snp$V5)

  # ---- 3. 1000G LD reference ----
  message(sprintf("  [%s] Loading 1000G BED for chr %s", gene.name, chr))
  kg_prefix <- file.path(ld_1kg_dir, paste0("1000G_b38_EUR_chr", chr))
  kg_geno   <- BEDMatrix(kg_prefix)

  kg_bim <- read.delim(paste0(kg_prefix, ".bim"), header = FALSE,
                       stringsAsFactors = FALSE)
  colnames(kg_geno) <- paste0("chr", kg_bim$V1, ":", kg_bim$V4, ":",
                              kg_bim$V6, ":", kg_bim$V5)

  # subset to gene region
  kg_gene <- kg_bim %>%
    mutate(idx = row_number()) %>%
    filter(V4 >= gene.start & V4 <= gene.end)

  if (nrow(kg_gene) == 0) {
    saveRDS("no 1KG SNPs in region", file.path(out_dir, paste0("nores_", gene.name, ".rds")))
    next
  }

  kg_mat <- data.matrix(kg_geno[, kg_gene$idx, drop = FALSE])

    kg_mat_raw <- data.matrix(kg_geno[, kg_gene$idx, drop = FALSE])
      kg_ids     <- paste0("chr", kg_gene$V1, ":", kg_gene$V4, ":", kg_gene$V6, ":", kg_gene$V5)
      colnames(kg_mat_raw) <- kg_ids
    
      # filter_X handles: missing imputation, MAF, zero-variance — then scale
      kg_mat <- filter_X(kg_mat_raw,
                         missing_rate_thresh = missing.cut,
                         maf_thresh          = maf.cut)
      kg_mat <- scale(kg_mat, center = TRUE, scale = TRUE)
    
      # sync kg_ref to variants that survived filter_X
      surviving <- colnames(kg_mat)
      kg_ref <- tibble(
        variant_id = surviving,
        chrom      = kg_gene$V1[match(surviving, kg_ids)],
        pos        = kg_gene$V4[match(surviving, kg_ids)],
        A2         = kg_gene$V6[match(surviving, kg_ids)],
        A1         = kg_gene$V5[match(surviving, kg_ids)]
      ) %>% select(-variant_id)

  # ---- 4. tissue loop ----
  eqtl_files <- list.files(eqtl_dir, pattern = "\\.bed\\.gz$")
  if (!is.null(tissues_considered))
    eqtl_files <- eqtl_files[str_detect(eqtl_files, paste(tissues_considered, collapse = "|"))]

  res_susie <- list()

  for (tissue.file in eqtl_files) {

    tissue <- str_extract(tissue.file, "^[^.]+")
    message(sprintf("[%s]   [%s] tissue: %s",
                    format(Sys.time(), "%Y-%m-%d %H:%M:%S"),
                    gene.name, tissue))

    # expression
    expr <- {
      path <- file.path(eqtl_dir, tissue.file)
      if (data.table) read_delim(path) else read.delim(path, stringsAsFactors = FALSE)
    }
    expr$gene_id <- sub("\\..*", "", expr$gene_id)
    expr <- filter(expr, gene_id == gene.name)
    if (nrow(expr) == 0) {
      message(sprintf("  [%s] No expression in %s - skipping", gene.name, tissue))
      next
    }

    samples <- colnames(expr)[-c(1:4)]
    if (!data.table) samples <- gsub("\\.", "-", samples)

    # residualize on covariates
    covar <- read_delim(file.path(cov_dir, paste0(tissue, ".v8.covariates.txt"))) %>%
      select(-ID) %>% as.matrix()
    covar        <- covar[, samples]
    covar_design <- cbind(1, t(covar))

    Y_resid <- .lm.fit(x = covar_design,
                       y = t(as.matrix(expr[, -c(1:4)])))$residuals
    X_raw   <- filter_X(geno_mat[samples, , drop = FALSE],
                        missing_rate_thresh = missing.cut,
                        maf_thresh          = maf.cut)
    X_resid <- .lm.fit(x = covar_design, y = X_raw)$residuals

    # marginal sumstats
    target_data <- calculate_sumstat(X_resid, Y_resid) %>%
      separate(SNP, into = c("chrom", "pos", "A2", "A1"), sep = ":", remove = FALSE) %>%
      mutate(chrom = str_remove(chrom, "chr"),
             chrom = as.numeric(chrom),
             pos   = as.numeric(pos))

    # harmonize 1000G genotype matrix to sumstat alleles
    harm <- sketch_harmonize(target_data  = target_data,
                             ref_variants = kg_ref,
                             geno_1kg     = kg_mat)
    if (ncol(harm$geno) == 0) {
      message(sprintf("  [%s] No variants survive harmonization in %s - skipping",
                      gene.name, tissue))
      next
    }

    # align individual X to harmonized variant set
    geno_idx <- match(colnames(harm$geno), colnames(X_resid))
    keep     <- !is.na(geno_idx)
    if (!all(keep)) {
      harm$geno    <- harm$geno[, keep, drop = FALSE]
      harm$sumstat <- harm$sumstat[keep, ]
      geno_idx     <- geno_idx[keep]
    }
    if (length(geno_idx) == 0) next
    X_update <- X_resid[, geno_idx, drop = FALSE]
    n        <- nrow(X_update)

    # compute LD from 1000G reference panel
    kg_LD <- crossprod(harm$geno) / (nrow(harm$geno) - 1)

    # kriging QC on 1000G-derived LD
    krig           <- kriging_fix(z = harm$sumstat$z, LD = kg_LD, n = n)
    harm$sumstat$z <- krig$z

    # ---- 5. fit models ----
    susie_res     <- susie(X = X_update, y = Y_resid)

    insample_sumstat <- calculate_sumstat(X_update, Y_resid)
    insample_z  <- insample_sumstat$z
    Xs          <- scale(X_update, center = TRUE, scale = TRUE)
    insample_LD <- crossprod(Xs) / (nrow(Xs) - 1)
        
      # ---- in-sample panel ----
      insample_res         <- safe_susie(susie_hc, "insample_res",
                                         z = insample_z, R = insample_LD, n = n)


      insample_eb_mix_res  <- safe_susie(susie_hc, "insample_eb_mix_res",
                                         z = insample_z, R = insample_LD, n = n,
                                         R_finite = B_insample, R_mismatch = "eb_mix")

      # ---- 1000G (KG) panel: kriged z against kg_LD ----
      susie_rss_res        <- safe_susie(susie_hc, "susie_rss_res",
                                         z = harm$sumstat$z, R = kg_LD, n = n)

      susie_eb_mix_res         <- safe_susie(susie_hc, "susie_eb_mix_res",
                                         z = harm$sumstat$z, R = kg_LD, n = n,
                                         R_finite = n_kg, R_mismatch = "eb_mix")

      susie_eb_mix_res_new <- safe_susie(
          susie_xw, "susie_eb_mix_res_new",
          z = harm$sumstat$z, R = kg_LD, n = n,
          R_finite = n_kg, R_mismatch = "eb_mix")


      susie_ser_res        <- safe_susie(susie_hc, "susie_ser_res",
                                         z = harm$sumstat$z, R = kg_LD, n = n,
                                         L = 1, max_iter = 1)
        
        # ---- max inter-CS correlation (NA when < 2 CS) ----
        # uses the LD each fit was run against; in-sample fits -> insample_LD,
        # sketch fits -> sketch_LD. Returns max |corr| among per-CS top-PIP variants.
    cs_maxcor <- function(model, Xcorr)
          tryCatch(as.numeric(get_cs_correlation(model, Xcorr = Xcorr, max = TRUE)),
                   error = function(e) NA_real_)
        
      max_cs_corr <- c(
        susie_res           = cs_maxcor(susie_res,            insample_LD),
        insample_res         = cs_maxcor(insample_res,         insample_LD),
        insample_eb_mix_res  = cs_maxcor(insample_eb_mix_res,  insample_LD),
        susie_rss_res        = cs_maxcor(susie_rss_res,        kg_LD),
        susie_eb_mix_res         = cs_maxcor(susie_eb_mix_res,         kg_LD),
        susie_eb_mix_res_new   = cs_maxcor(susie_eb_mix_res_new,   kg_LD),
        susie_ser_res        = cs_maxcor(susie_ser_res,        kg_LD)
      )
        
     res_susie[[tissue]] <- list(
          variant_ids          = colnames(harm$sketch),
          # --- fits ---
          susie_res            = susie_res,
          insample_res         = insample_res,
          insample_eb_mix_res  = insample_eb_mix_res,
          susie_rss_res        = susie_rss_res,
          susie_eb_mix_res     = susie_eb_mix_res,
          susie_eb_mix_res_new = susie_eb_mix_res_new,
          susie_ser_res        = susie_ser_res,
          # --- diagnostics / bookkeeping ---
          max_cs_corr          = max_cs_corr,
          sample_size          = n,
          flipped_prop         = harm$flipped_prop,
          krig_outliers        = krig$outlier_number
        )
      }

  # ---- 6. save per-gene ----
  if (length(res_susie) == 0) {
    saveRDS("no tissues processed",
            file.path(out_dir, paste0("nores_", gene.name, ".rds")))
  } else {
    saveRDS(res_susie,
            file.path(out_dir, paste0("res_", gene.name, ".rds")))
  }
}

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/code/Gtex_fine.R


In [9]:
%%writefile /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/code/submit_Gtex_fine.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J "Gtex_fine[1-2]"
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 72:00
#BSUB -o /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/log/Gtex_fine.%J.%I.out
#BSUB -e /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/log/Gtex_fine.%J.%I.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

workdir=/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/
mkdir -p ${workdir}/log
code_dir="$workdir/code"
tmp_root="$workdir/tmp"

tmp_dir="$tmp_root/${LSB_JOBID}_${LSB_JOBINDEX}"
mkdir -p "$tmp_dir"

export TMPDIR="$tmp_dir"
export TMP="$tmp_dir"
export TEMP="$tmp_dir"

Rscript "$code_dir/Gtex_fine.R" "$LSB_JOBINDEX"

Overwriting /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/code/submit_Gtex_fine.sh


In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G//"
job_name="coloc"
job_dir="${base_dir}/${job_name}"

# Source: where the per-gene SuSiE result RDS files live
src_dir="${base_dir}/res"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=25000]
#BSUB -W 36:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/16.Fig5A_5C.GTEx_Coloc.ipynb" coloc \
    --in-dir SRC_DIR \
    --out-dir WORK_DIR/result/ \
    --mem 40G -J 50
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}.sh.submit"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|SRC_DIR|${src_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    "${base_sh}" > "${output_script}"
 sb "${output_script}"